In [1]:
import numpy as np
import math
from scipy.stats import norm, beta
from scipy.integrate import quad
from scipy.special import expit, betaln, betainc
from numpy.polynomial.hermite import hermgauss
from numpy.polynomial.legendre import leggauss
import matplotlib.pyplot as plt
import pandas as pd
from scipy.fft import rfft, next_fast_len
from scipy import stats
from mpl_toolkits.axes_grid1 import make_axes_locatable



In [2]:

def logit_transform(f, eps=1e-12):
    f = np.clip(f, eps, 1 - eps)
    return np.log(f / (1.0 - f))


# ======================================================================
# Data generation
# ======================================================================

def simulate_isnv_frequency_init(
    t,
    f2_0=0.15,
    epsilon=0.2,
    rho=0.1,
    seed=None,
    return_X=False,
    init_dist="exponential",
    true_floor=0.0,
    max_resample=100000,
):
    """
    Simulated iSNV frequency dynamics using dX(t) = sigma_X dB(t),
    with sigma_X = epsilon * sqrt(2 * (1 - rho)) and f2 = logistic(X).

    NOTE on the initial frequency: the TRUE initial is drawn from the plain
    generative distribution (Exponential(mean f2_0)) and resampled ONLY to keep
    it a valid frequency in (true_floor, 1). With true_floor=0 there is no
    biological lower truncation: true frequencies below the calling threshold do
    occur, and whether such a variant enters a dataset is decided later, by the
    measurement + observed-initial filter (cut B), not by truncating the truth.
    The calling threshold therefore lives on the OBSERVED side only
    (keep_initial_in_range / the final-censoring threshold), never here.

    The Brownian increments are drawn in one batch; numpy's Generator produces
    the identical stream as the previous per-step scalar draws, so trajectories
    are bit-for-bit unchanged for a given seed (just much faster, which matters
    now that we oversample).
    """
    t = np.asarray(t, dtype=float)
    if t.size < 2:
        raise ValueError("t must contain at least two time points.")
    if np.any(np.diff(t) <= 0):
        raise ValueError("t must be strictly increasing.")
    if not (-1.0 <= rho <= 1.0):
        raise ValueError("rho must be in [-1, 1].")
    if epsilon < 0:
        raise ValueError("epsilon must be nonnegative.")
    if not (0.0 <= true_floor < 1.0):
        raise ValueError("true_floor must be in [0, 1).")

    rng = np.random.default_rng(seed)

    def draw_initial_frequency():
        if init_dist in (None, "fixed"):
            return float(f2_0)
        if init_dist == "exponential":
            if f2_0 <= 0:
                raise ValueError("For init_dist='exponential', f2_0 (mean) must be > 0.")
            return float(rng.exponential(scale=f2_0))
        raise ValueError("init_dist must be one of {'exponential','fixed',None}.")

    # resample ONLY to keep a valid frequency in (true_floor, 1); with
    # true_floor=0 this rejects only the rare exponential draw that lands >= 1.
    f2_init = draw_initial_frequency()
    tries = 0
    while not (true_floor < f2_init < 1.0):
        tries += 1
        if tries >= max_resample:
            raise RuntimeError("Exceeded max_resample while drawing initial frequency.")
        f2_init = draw_initial_frequency()

    X0 = np.log(f2_init / (1.0 - f2_init))
    sigma_X = epsilon * np.sqrt(2.0 * (1.0 - rho))

    dt = np.diff(t)
    dB = rng.standard_normal(dt.size) * np.sqrt(dt)
    X = np.empty(t.size, dtype=float)
    X[0] = X0
    X[1:] = X0 + sigma_X * np.cumsum(dB)

    f2 = 1.0 / (1.0 + np.exp(-X))
    if return_X:
        return f2, X
    return f2


def generate_dataset(t, epsilon, rho, n_traj=500, base_seed=1234, f2_mean=0.15,
                     true_floor=0.0):
    n_time = len(t)
    data = np.empty((n_traj, n_time))
    for i in range(n_traj):
        seed_i = base_seed + i
        data[i, :] = simulate_isnv_frequency_init(
            t=t, f2_0=f2_mean, epsilon=epsilon, rho=rho, seed=seed_i,
            return_X=False, init_dist="exponential", true_floor=true_floor,
        )
    return data


def sample_initial_final(dataset, t, T, atol_time=1e-8):
    t = np.asarray(t, dtype=float)
    if dataset.ndim != 2:
        raise ValueError("dataset must be 2D array: (n_traj, n_time).")
    n_traj, n_time = dataset.shape
    if n_time != t.size:
        raise ValueError("dataset number of timepoints does not match length of t.")
    if not (t.min() - atol_time <= T <= t.max() + atol_time):
        print(f"Requested T={T} is outside simulated time range [{t.min()}, {t.max()}].")
        return None
    idx_T = int(np.argmin(np.abs(t - T)))
    samples = np.empty((n_traj, 2), dtype=float)
    samples[:, 0] = dataset[:, 0]
    samples[:, 1] = dataset[:, idx_T]
    return samples


def add_beta_measurement_noise(dataset, nu_measnoise=1.0, seed=None, eps=1e-12):
    """f_obs ~ Beta(mu*nu, (1-mu)*nu), applied to every timepoint (untruncated)."""
    dataset = np.asarray(dataset, dtype=float)
    if nu_measnoise <= 0:
        raise ValueError("nu_measnoise must be positive.")
    rng = np.random.default_rng(seed)
    mu = np.clip(dataset, eps, 1.0 - eps)
    return rng.beta(mu * nu_measnoise, (1.0 - mu) * nu_measnoise)


# ======================================================================
# Observed-initial filter
# ======================================================================
# The measurement process draws an *untruncated* beta and we then discard
# trajectories whose OBSERVED initial frequency falls outside the calling
# range. This is a filter on the support of the data, NOT a truncated emission
# density -- so the inference below uses the plain (untruncated) beta.
#
# Note the filter depends only on column 0 of the noisy dataset, which is the
# same for every T. So we can select the surviving trajectories ONCE and reuse
# the identical 50 trajectories at every T (better for cross-T comparison).

def keep_initial_in_range(samples, lower=0.02, upper=0.98):
    """Keep only pairs whose observed initial frequency (col 0) is in [lower, upper]."""
    if samples is None:
        return None
    keep = (samples[:, 0] >= lower) & (samples[:, 0] <= upper)
    return samples[keep]


def select_calibrated_trajectories(noisy_dataset, lower=0.02, upper=0.98, n_keep=50,
                                   label=""):
    """
    Oversampled -> filtered -> trimmed to exactly n_keep trajectories.

    Returns (dataset_kept, idx_kept). Raises if fewer than n_keep survive, with a
    message telling you how much to oversample.
    """
    q0_obs = noisy_dataset[:, 0]
    idx = np.flatnonzero((q0_obs >= lower) & (q0_obs <= upper))
    n_gen = noisy_dataset.shape[0]
    if idx.size < n_keep:
        rate = idx.size / n_gen
        need = int(math.ceil(n_keep / max(rate, 1e-9) * 1.2))
        raise RuntimeError(
            f"{label}: only {idx.size}/{n_gen} trajectories passed the initial filter "
            f"(rate {rate:.2f}); need {n_keep}. Increase N_TRAJ_GEN to about {need}."
        )
    n_pass = idx.size
    idx = idx[:n_keep]                      # keep the first n_keep survivors
    print(f"{label}: {n_pass}/{n_gen} passed the initial filter "
          f"(rate {n_pass / n_gen:.2f}); using the first {n_keep}.")
    return noisy_dataset[idx], idx


# ======================================================================
# Build datasets, add noise, select exactly N_KEEP survivors, then pair up
# ======================================================================

t_start, t_end, dt = 0.0, 10.0, 0.001
t = np.arange(t_start, t_end + dt, dt)

N_KEEP = 50        # number of mock pairs we actually want to fit
N_TRAJ_GEN = 200   # oversample: ~55-65% survive the initial filter at nu = 10

dataset_sigma_06_small = generate_dataset(t, epsilon=0.6, rho=0.5, n_traj=N_TRAJ_GEN, base_seed=200)
dataset_sigma_10_small = generate_dataset(t, epsilon=1.0, rho=0.5, n_traj=N_TRAJ_GEN, base_seed=100)
dataset_sigma_14_small = generate_dataset(t, epsilon=1.4, rho=0.5, n_traj=N_TRAJ_GEN, base_seed=400)

nu_measnoise = 10  # true measurement concentration used to build the mock data

dataset_sigma_06_measnoise = add_beta_measurement_noise(dataset_sigma_06_small, nu_measnoise, seed=1200)
dataset_sigma_10_measnoise = add_beta_measurement_noise(dataset_sigma_10_small, nu_measnoise, seed=1100)
dataset_sigma_14_measnoise = add_beta_measurement_noise(dataset_sigma_14_small, nu_measnoise, seed=1400)

datasets_map_measnoise = {
    "sigma_06": dataset_sigma_06_measnoise,
    "sigma_10": dataset_sigma_10_measnoise,
    "sigma_14": dataset_sigma_14_measnoise,
}

# calling range used to FILTER the observed initial
INIT_FILTER_LOWER = 0.02
INIT_FILTER_UPPER = 0.98
# threshold used to CENSOR the final observation inside the measurement model
FINAL_CENSOR_THRESHOLD = 0.02

T_hours_list = [24, 48, 96]
T_days_map = {h: h / 24.0 for h in T_hours_list}

datasets_kept = {}
X_samples_all_measnoise = {}
for short, dataset in datasets_map_measnoise.items():
    kept, _ = select_calibrated_trajectories(
        dataset, INIT_FILTER_LOWER, INIT_FILTER_UPPER, N_KEEP, label=short
    )
    datasets_kept[short] = kept
    X_samples_all_measnoise[short] = {}
    for T_hours in T_hours_list:
        samples = sample_initial_final(dataset=kept, t=t, T=T_days_map[T_hours])
        X_samples_all_measnoise[short][T_hours] = None if samples is None else logit_transform(samples)


# ======================================================================
# Likelihood
# ----------------------------------------------------------------------
# Conditional on the observed initial q0_obs, marginalising over BOTH the
# latent true initial q0_true and the latent true final qT_true:
#
#   p(qT_obs | q0_obs) =
#       [ int int  g_q(qT_obs|qT_true) f_q(qT_true|q0_true) g0(q0_obs|q0_true) pi(q0_true) dqT dq0 ]
#       -----------------------------------------------------------------------------------------
#                        [ int g0(q0_obs|u) pi(u) du ]                         (= evidence Z)
#
# For real data pi (the generative distribution of true initial frequencies) is
# unknown, so prior_mode lets you drop it:
#
#   "exponential" : pi propto exp(-q0/f2_mean) on [lower_true, upper_true)  -- the true prior
#   "flat"        : pi propto 1 on [lower_true, upper_true)                  -- prior-free approx
#   "flat_logit"  : pi propto 1/(q0(1-q0)), i.e. flat in X-space (improper at
#                   the edges; use a finite upper_true such as 1 - lower_true)
#   "plugin"      : ignore initial measurement noise entirely, X0_true := X0_obs
#                   (no outer integral at all -- the cheapest approximation)
#
# Note pi only enters through the *posterior for q0_true given q0_obs*; it does
# not cancel, because the numerator reweights that posterior by the dynamics.
# ======================================================================

# Lower bound of the prior on the TRUE initial frequency, used both as the
# support cutoff of pi and as the lower limit of the outer q0_true integral.
# It should MATCH the generative support: since the simulator now draws the true
# initial from an untruncated exponential (true_floor=0), this is just a small
# numerical floor so that X0 = logit(q0) stays finite. Results are insensitive
# to its exact value (the integrand vanishes as q0_true -> 0 for any q0_obs>0).
TRUE_INIT_FLOOR = 1e-4

def _beta_pdf(x, a, b):
    """Beta pdf via log form (fast, vectorised). x scalar, a,b arrays."""
    return np.exp((a - 1.0) * math.log(x) + (b - 1.0) * math.log1p(-x) - betaln(a, b))


def _initial_prior(q0_true, f2_mean=0.15, lower_true=TRUE_INIT_FLOOR, mode="exponential"):
    """Unnormalised prior on the true initial frequency (normaliser cancels in num/Z)."""
    q0_true = np.asarray(q0_true, dtype=float)
    if mode == "exponential":
        p = np.exp(-q0_true / f2_mean)
    elif mode in ("flat", "uniform", "none"):
        p = np.ones_like(q0_true)
    elif mode == "flat_logit":
        p = 1.0 / (q0_true * (1.0 - q0_true))
    else:
        raise ValueError("mode must be one of {'exponential','flat','flat_logit'}.")
    return np.where(q0_true >= lower_true, p, 0.0)


# --- scalar reference implementation (transparent, for validation) --------

def one_pair_loglik_reference(
    q0_obs, qT_obs, T, sigma_x, nu,
    final_threshold=0.02, f2_mean=0.15, lower_true=TRUE_INIT_FLOOR, upper_true=1.0 - 1e-9,
    prior_mode="exponential", eps=1e-300,
):
    """Direct double integral with scipy.quad -- mirrors the equations 1:1."""
    lower, upper = final_threshold, 1.0 - final_threshold
    sd = sigma_x * math.sqrt(T)

    def final_meas(qT_true):
        a, b = nu * qT_true, nu * (1.0 - qT_true)
        if qT_obs < lower:
            return betainc(a, b, lower)
        if qT_obs > upper:
            return 1.0 - betainc(a, b, upper)
        return _beta_pdf(qT_obs, a, b)

    def inner(X0_true):
        # int g_q(qT_obs|qT_true) f_q(qT_true|X0_true) dqT_true, done in X-space
        f = lambda XT: final_meas(np.clip(expit(XT), 1e-12, 1 - 1e-12)) * norm.pdf(XT, X0_true, sd)
        val, _ = quad(f, X0_true - 12 * sd, X0_true + 12 * sd, limit=200)
        return val

    if prior_mode == "plugin":
        return math.log(max(inner(math.log(q0_obs / (1.0 - q0_obs))), eps))

    def w(q0_true):  # g0(q0_obs|q0_true) * pi(q0_true)
        a, b = nu * q0_true, nu * (1.0 - q0_true)
        return _beta_pdf(q0_obs, a, b) * float(
            _initial_prior(np.array(q0_true), f2_mean, lower_true, prior_mode)
        )

    num, _ = quad(lambda q: w(q) * inner(math.log(q / (1.0 - q))),
                  lower_true, upper_true, limit=200)
    Z, _ = quad(w, lower_true, upper_true, limit=200)
    return math.log(max(num / Z, eps))


# --- vectorised grid implementation (Gauss-Hermite inner, Gauss-Legendre outer) ---

def _censored_inner(X0_nodes, sd, nu, qT_obs, below, above, inrng, xh, wh,
                    lower, upper, gh_norm):
    """
    Gauss-Hermite inner integral E_{X_T ~ N(X0, sd^2)}[ g(qT_obs | X_T) ].

    X0_nodes has shape (M,) -> returns (P, M).
    """
    P = qT_obs.size
    XT = X0_nodes[:, None] + math.sqrt(2.0) * sd * xh[None, :]   # (M, K)
    qT = np.clip(expit(XT), 1e-12, 1.0 - 1e-12)
    aT, bT = nu * qT, nu * (1.0 - qT)

    I = np.zeros((P, X0_nodes.size))
    if inrng.any():
        lnB_T = betaln(aT, bT)
        qo = qT_obs[inrng]
        pdf = np.exp((aT[None] - 1.0) * np.log(qo)[:, None, None]
                     + (bT[None] - 1.0) * np.log1p(-qo)[:, None, None]
                     - lnB_T[None])
        I[inrng] = gh_norm * (pdf @ wh)
    if below.any():
        I[below] = gh_norm * (betainc(aT, bT, lower) @ wh)
    if above.any():
        I[above] = gh_norm * ((1.0 - betainc(aT, bT, upper)) @ wh)
    return I


def loglik_grid(
    X_pairs, T, sigma_grid, nu_grid,
    final_threshold=0.02, f2_mean=0.15, lower_true=TRUE_INIT_FLOOR, upper_true=1.0 - 1e-9,
    prior_mode="exponential", normalise_posterior=True,
    n_outer=48, n_hermite=24, eps=1e-300,
):
    """
    Total log-likelihood over all pairs on a (sigma_x, nu) grid.

    Inner Gaussian integral over X_T -> Gauss-Hermite.
    Outer integral over q0_true in [lower_true, upper_true) -> Gauss-Legendre
    (skipped entirely when prior_mode == 'plugin').

    prior_mode: 'exponential' | 'flat' | 'flat_logit' | 'plugin'

    normalise_posterior=False takes the weight g0*pi literally as the posterior
    without dividing by its integral Z. With prior_mode='flat' this is exactly
    the ad hoc "p(q0_true|q0_obs) := g0(q0_obs|q0_true)" approximation. It is not
    a probability density (its mass depends on q0_obs and on nu), so it perturbs
    nu somewhat; in practice it leaves sigma_x essentially unchanged.
    """
    lower, upper = final_threshold, 1.0 - final_threshold

    q0_obs = expit(X_pairs[:, 0])            # (P,)
    qT_obs = expit(X_pairs[:, 1])            # (P,)

    # censoring status of the FINAL observation (fixed per pair)
    below = qT_obs < lower
    above = qT_obs > upper
    inrng = ~(below | above)

    xh, wh = hermgauss(n_hermite)
    gh_norm = 1.0 / math.sqrt(math.pi)

    logL = np.empty((sigma_grid.size, nu_grid.size))

    # ---- cheapest approximation: no latent initial at all -----------------
    if prior_mode == "plugin":
        X0_obs = X_pairs[:, 0]
        for j, nu in enumerate(nu_grid):
            for i, sigma_x in enumerate(sigma_grid):
                sd = sigma_x * math.sqrt(T)
                XT = X0_obs[:, None] + math.sqrt(2.0) * sd * xh[None, :]   # (P, K)
                qT = np.clip(expit(XT), 1e-12, 1.0 - 1e-12)
                aT, bT = nu * qT, nu * (1.0 - qT)
                val = np.zeros(X0_obs.size)
                if inrng.any():
                    a_i, b_i = aT[inrng], bT[inrng]
                    qo = qT_obs[inrng][:, None]
                    pdf = np.exp((a_i - 1.0) * np.log(qo) + (b_i - 1.0) * np.log1p(-qo)
                                 - betaln(a_i, b_i))
                    val[inrng] = gh_norm * (pdf @ wh)
                if below.any():
                    val[below] = gh_norm * (betainc(aT[below], bT[below], lower) @ wh)
                if above.any():
                    val[above] = gh_norm * ((1.0 - betainc(aT[above], bT[above], upper)) @ wh)
                logL[i, j] = np.log(np.maximum(val, eps)).sum()
        return logL

    # ---- marginalise over the latent true initial -------------------------
    xl, wl = leggauss(n_outer)
    a_, b_ = lower_true, upper_true
    q0t = 0.5 * (b_ - a_) * xl + 0.5 * (b_ + a_)      # (M,)
    Wl = 0.5 * (b_ - a_) * wl                          # (M,)
    X0t = np.log(q0t / (1.0 - q0t))                    # (M,)
    prior0 = _initial_prior(q0t, f2_mean, lower_true, prior_mode)   # (M,)

    for j, nu in enumerate(nu_grid):
        # initial-measurement pieces (depend on nu, q0t; shared across sigma & pairs)
        a0, b0 = nu * q0t, nu * (1.0 - q0t)            # (M,)
        lnB0 = betaln(a0, b0)                          # (M,)
        g0 = np.exp((a0[None, :] - 1.0) * np.log(q0_obs)[:, None]
                    + (b0[None, :] - 1.0) * np.log1p(-q0_obs)[:, None]
                    - lnB0[None, :])                   # (P, M)
        wgt = g0 * prior0[None, :]                     # (P, M) unnormalised posterior weight
        Z = (wgt @ Wl) if normalise_posterior else np.ones(q0_obs.size)

        for i, sigma_x in enumerate(sigma_grid):
            sd = sigma_x * math.sqrt(T)
            I = _censored_inner(X0t, sd, nu, qT_obs, below, above, inrng,
                                xh, wh, lower, upper, gh_norm)     # (P, M)
            num = (wgt * I) @ Wl                                   # (P,)
            logL[i, j] = np.log(np.maximum(num / Z, eps)).sum()

    return logL


def loglik_grid_free_prior_mean(
    X_pairs, T, sigma_grid, nu_grid, f2_mean_grid, **kw
):
    """
    Middle ground between knowing pi and dropping it: keep the exponential shape
    but treat its mean as a third unknown parameter, estimated from the same data.

    Returns an array of shape (n_sigma, n_nu, n_f2mean).
    """
    kw.pop("prior_mode", None)
    kw.pop("f2_mean", None)
    out = np.empty((sigma_grid.size, nu_grid.size, f2_mean_grid.size))
    for m, f2m in enumerate(f2_mean_grid):
        out[:, :, m] = loglik_grid(
            X_pairs, T, sigma_grid, nu_grid, prior_mode="exponential", f2_mean=f2m, **kw
        )
    return out


# ======================================================================
# Fit on the sigma_06 dataset, T = 24 h  (truth: sigma_x = 0.6, nu = 10)
# ======================================================================

def prior_influence_diagnostic(X_pairs, T, sigma_x, nu, verbose=True):
    """
    How much can pi matter? Compare the diffusion scale over the interval,
    sigma_x*sqrt(T), with the logit-space sd of the initial measurement noise
    (delta method: sd_logit ~ 1/sqrt((nu+1) q0 (1-q0))).

    r >> 1 : the dynamics dominate, the latent initial is pinned down by the data
             and the choice of pi is nearly irrelevant -> drop it safely.
    r <~ 1 : q0_true is poorly determined by its own observation, so whatever pi
             you assume fills the gap -> dropping it biases sigma_x upward.
    """
    q0 = expit(np.asarray(X_pairs)[:, 0])
    sd_meas_logit = 1.0 / np.sqrt((nu + 1.0) * q0 * (1.0 - q0))
    r = sigma_x * math.sqrt(T) / np.median(sd_meas_logit)
    if verbose:
        print(f"  prior-influence ratio r = sigma*sqrt(T) / sd_logit(meas) = {r:.2f} "
              f"({'pi largely irrelevant' if r > 1.5 else 'pi matters -- do not drop it'})")
    return r


def summarise(logL, sigma_grid, nu_grid, tag):
    k = np.unravel_index(np.argmax(logL), logL.shape)
    print(f"{tag:>28s}:  sigma_x_hat = {sigma_grid[k[0]]:.3f} (truth 0.6),  "
          f"nu_hat = {nu_grid[k[1]]:.2f} (truth 10)")
    return sigma_grid[k[0]], nu_grid[k[1]]


sigma_06: 150/200 passed the initial filter (rate 0.75); using the first 50.
sigma_10: 145/200 passed the initial filter (rate 0.72); using the first 50.
sigma_14: 139/200 passed the initial filter (rate 0.69); using the first 50.


In [ ]:

if __name__ == "__main__":
    sigma_grid = np.linspace(0.05, 2.2, 60)
    nu_grid = np.logspace(-1, 4, 60)

    X_pairs = X_samples_all_measnoise["sigma_06"][24]
    T_days = 24 / 24.0
    print(f"\nFitting {X_pairs.shape[0]} pairs at T = 24 h\n")

    results = {}
    for mode in ("exponential", "flat", "plugin"):
        logL = loglik_grid(
            X_pairs, T_days, sigma_grid, nu_grid,
            final_threshold=FINAL_CENSOR_THRESHOLD, prior_mode=mode,
        )
        results[mode] = logL
        s_hat, nu_hat = summarise(logL, sigma_grid, nu_grid, f"prior_mode='{mode}'")
        if mode == "exponential":
            prior_influence_diagnostic(X_pairs, T_days, s_hat, nu_hat)
        np.save(f"heatmap_results/mock_sigma_06_logL_{mode}.npy", logL)

    # pi kept parametric, its mean estimated jointly
    f2_mean_grid = np.linspace(0.04, 0.50, 24)
    logL3 = loglik_grid_free_prior_mean(
        X_pairs, T_days, sigma_grid, nu_grid, f2_mean_grid,
        final_threshold=FINAL_CENSOR_THRESHOLD,
    )
    k = np.unravel_index(np.argmax(logL3), logL3.shape)
    print(f"{'free prior mean':>28s}:  sigma_x_hat = {sigma_grid[k[0]]:.3f} (truth 0.6),  "
          f"nu_hat = {nu_grid[k[1]]:.2f} (truth 10),  "
          f"f2_mean_hat = {f2_mean_grid[k[2]]:.3f} (truth 0.15)")
    np.save("heatmap_results/mock_sigma_06_logL_freemean.npy", logL3)

In [4]:
"""
Replacement for the double-integral conditional likelihood.

Both integrals are moved onto ONE fixed uniform grid in logit space, and the
inner (latent-final) integral is recognised as a Gaussian convolution, which is
then done exactly in Fourier space. Nothing about the quadrature depends on
sigma_x, so the log-likelihood surface is smooth by construction.

    p(qT_obs | q0_obs) = num / Z

    num = int w(X0) * S(X0) dX0,     S = ( g_T  *  phi_sd )   (convolution)
    Z   = int w(X0) dX0
    w(X0) = g_0(q0_obs | expit(X0)) * pi(expit(X0)) * expit(X0)(1-expit(X0))

The Jacobian factor q(1-q) appears because pi is a density in q while the grid
and the Brownian transition live in X = logit(q).

Cost per (sigma, nu): one matrix-vector product. The beta evaluations -- the
expensive part -- are hoisted out of the sigma loop entirely.
"""

__all__ = ["LogitGrid", "make_grid", "loglik_grid_spectral", "inner_integral_grid"]


# ----------------------------------------------------------------------
# Grid construction
# ----------------------------------------------------------------------

class LogitGrid:
    """Uniform grid X_k on [x_lo, x_hi] with spacing h, plus its rfft plan size."""

    def __init__(self, x_lo, x_hi, h):
        self.K = int(math.ceil((x_hi - x_lo) / h)) + 1
        self.h = h
        self.x_lo = x_lo
        self.X = x_lo + h * np.arange(self.K)
        self.q = expit(self.X)
        # zero-pad to at least 2K so the circular convolution is a linear one
        self.N = next_fast_len(2 * self.K)
        self.nfreq = self.N // 2 + 1
        self.omega = 2.0 * math.pi * np.arange(self.nfreq) / (self.N * h)
        # Hermitian-symmetry multipliers for a full-spectrum Parseval sum
        self.mult = np.full(self.nfreq, 2.0)
        self.mult[0] = 1.0
        if self.N % 2 == 0:
            self.mult[-1] = 1.0

    def __repr__(self):
        return (f"LogitGrid(range=[{self.x_lo:.1f}, {self.X[-1]:.1f}], "
                f"h={self.h:.2e}, K={self.K}, N_fft={self.N})")


def make_grid(nu_max, sigma_max, T, sigma_min=None, points_per_width=8.0,
              x_core=(-12.0, 12.0), pad_sds=9.0, verbose=False):
    """
    Choose grid spacing and extent from the two length scales in the problem.

    Resolution is set by the narrowest feature: the beta measurement likelihood,
    whose width in logit space is ~1/sqrt(nu*q(1-q)) >= 2/sqrt(nu).
    Extent is set by the widest: the Brownian kernel, sd = sigma_max*sqrt(T),
    padded so the tails of censored (non-decaying) integrands are negligible.
    """
    sd_max = sigma_max * math.sqrt(T)
    h = (2.0 / math.sqrt(nu_max)) / points_per_width
    if sigma_min is not None:
        sd_min = sigma_min * math.sqrt(T)
        if sd_min < 4.0 * h:
            h = sd_min / 4.0  # also resolve the narrowest Brownian kernel
    g = LogitGrid(x_core[0] - pad_sds * sd_max, x_core[1] + pad_sds * sd_max, h)
    if verbose:
        print(f"  {g}")
    return g


# ----------------------------------------------------------------------
# Per-nu precomputation (independent of sigma_x)
# ----------------------------------------------------------------------

def _log_final_emission(qT_obs, q, nu, lower, upper):
    """(P, K) log g(qT_obs | q_k), censored below `lower` and above `upper`."""
    a, b = nu * q, nu * (1.0 - q)
    lnB = betaln(a, b)
    below, above = qT_obs < lower, qT_obs > upper
    inr = ~(below | above)
    out = np.empty((qT_obs.size, q.size))
    if inr.any():
        y = qT_obs[inr][:, None]
        out[inr] = (a - 1.0) * np.log(y) + (b - 1.0) * np.log1p(-y) - lnB
    if below.any():
        out[below] = np.log(np.maximum(betainc(a, b, lower), 1e-300))
    if above.any():
        # 1 - I_x(a,b) = I_{1-x}(b,a): the symmetric form keeps precision
        out[above] = np.log(np.maximum(betainc(b, a, 1.0 - upper), 1e-300))
    return out


def _log_initial_weight(q0_obs, q, nu, f2_mean, lower_true, upper_true, prior_mode):
    """(P, K) log[ g_0(q0_obs | q_k) * pi(q_k) * q_k(1-q_k) ], -inf outside support."""
    a, b = nu * q, nu * (1.0 - q)
    y = q0_obs[:, None]
    log_g0 = (a - 1.0) * np.log(y) + (b - 1.0) * np.log1p(-y) - betaln(a, b)

    with np.errstate(divide="ignore"):
        log_jac = np.log(q) + np.log1p(-q)
        if prior_mode == "exponential":
            log_pi = -q / f2_mean
        elif prior_mode in ("flat", "uniform", "none"):
            log_pi = np.zeros_like(q)
        elif prior_mode == "flat_logit":
            log_pi = -log_jac  # flat in X; the Jacobian cancels exactly
        else:
            raise ValueError("prior_mode must be one of "
                             "{'exponential','flat','flat_logit','plugin'}.")

    row = log_pi + log_jac
    row = np.where((q >= lower_true) & (q < upper_true), row, -np.inf)
    return log_g0 + row[None, :]


def _rows_to_spectrum(logM, grid):
    """Exponentiate rows with a per-row max removed, zero-pad, rfft."""
    off = np.max(logM, axis=1)
    off = np.where(np.isfinite(off), off, 0.0)
    M = np.exp(logM - off[:, None])
    M[~np.isfinite(M)] = 0.0
    pad = np.zeros((M.shape[0], grid.N))
    pad[:, :grid.K] = M
    return rfft(pad, axis=1), off


class _NuCache:
    """Everything that depends on nu but not on sigma_x."""

    def __init__(self, X_pairs, nu, grid, final_threshold, f2_mean,
                 lower_true, upper_true, prior_mode, normalise_posterior):
        q0_obs = expit(X_pairs[:, 0])
        qT_obs = expit(X_pairs[:, 1])
        lower, upper = final_threshold, 1.0 - final_threshold

        logG = _log_final_emission(qT_obs, grid.q, nu, lower, upper)
        Ghat, self.g_off = _rows_to_spectrum(logG, grid)

        if prior_mode == "plugin":
            # collapse the outer integral to a point mass at X0_obs, implemented
            # as a linear-interpolation delta so the same machinery applies
            W = np.zeros((q0_obs.size, grid.K))
            pos = (X_pairs[:, 0] - grid.x_lo) / grid.h
            i0 = np.clip(np.floor(pos).astype(int), 0, grid.K - 2)
            frac = pos - i0
            W[np.arange(W.shape[0]), i0] = (1.0 - frac) / grid.h
            W[np.arange(W.shape[0]), i0 + 1] = frac / grid.h
            pad = np.zeros((W.shape[0], grid.N))
            pad[:, :grid.K] = W
            What = rfft(pad, axis=1)
            self.w_off = np.zeros(q0_obs.size)
            self.log_Z = np.zeros(q0_obs.size)
        else:
            logW = _log_initial_weight(q0_obs, grid.q, nu, f2_mean,
                                       lower_true, upper_true, prior_mode)
            What, self.w_off = _rows_to_spectrum(logW, grid)
            # Zs is computed from the *scaled* rows, so log Z = log Zs + w_off
            Zs = grid.h * np.exp(logW - self.w_off[:, None]).sum(axis=1)
            self.log_Z = (np.log(Zs) + self.w_off if normalise_posterior
                          else np.zeros_like(Zs))

        # Parseval kernel: num = (h/N) * sum_j mult_j * Re[What_j conj(Ghat_j)] * H_j
        self.C = (grid.mult[None, :] *
                  np.real(What * np.conj(Ghat))) * (grid.h / grid.N)
        # constant offset that survives the num/Z ratio
        self.log_off = self.g_off + self.w_off - self.log_Z


# ----------------------------------------------------------------------
# Main entry point -- drop-in for loglik_grid
# ----------------------------------------------------------------------

def loglik_grid_spectral(
    X_pairs, T, sigma_grid, nu_grid,
    final_threshold=0.02, f2_mean=0.15, lower_true=1e-4, upper_true=1.0 - 1e-9,
    prior_mode="exponential", normalise_posterior=True,
    grid=None, points_per_width=8.0, eps=1e-300, return_grid=False,
):
    """
    Total log-likelihood over all pairs on a (sigma_x, nu) grid.

    Signature matches loglik_grid; `n_outer`/`n_hermite` are replaced by the
    single `points_per_width` knob (raise it to tighten the tolerance).
    """
    X_pairs = np.asarray(X_pairs, dtype=float)
    sigma_grid = np.asarray(sigma_grid, dtype=float)
    nu_grid = np.asarray(nu_grid, dtype=float)

    if grid is None:
        grid = make_grid(nu_grid.max(), sigma_grid.max(), T,
                         sigma_min=sigma_grid.min(),
                         points_per_width=points_per_width)

    # transfer function of the Brownian kernel, all sigmas at once: (nfreq, S)
    sd = sigma_grid * math.sqrt(T)
    H = np.exp(-0.5 * np.outer(grid.omega ** 2, sd ** 2))

    logL = np.empty((sigma_grid.size, nu_grid.size))
    for j, nu in enumerate(nu_grid):
        cache = _NuCache(X_pairs, nu, grid, final_threshold, f2_mean,
                         lower_true, upper_true, prior_mode, normalise_posterior)
        ratio = cache.C @ H                        # (P, S)
        logL[:, j] = np.log(np.maximum(ratio, eps)).sum(axis=0) \
            + cache.log_off.sum()

    return (logL, grid) if return_grid else logL


def inner_integral_grid(X0, qT_obs, sigma_x, T, nu, final_threshold=0.02,
                        grid=None, points_per_width=8.0):
    """Standalone inner integral E[g(qT_obs | X_T)], for validation."""
    X0 = np.atleast_1d(np.asarray(X0, float))
    if grid is None:
        grid = make_grid(nu, sigma_x, T, sigma_min=sigma_x,
                         points_per_width=points_per_width)
    lower, upper = final_threshold, 1.0 - final_threshold
    logG = _log_final_emission(np.atleast_1d(qT_obs), grid.q, nu, lower, upper)
    Ghat, off = _rows_to_spectrum(logG, grid)
    H = np.exp(-0.5 * (grid.omega * sigma_x * math.sqrt(T)) ** 2)

    W = np.zeros((X0.size, grid.K))
    pos = (X0 - grid.x_lo) / grid.h
    i0 = np.clip(np.floor(pos).astype(int), 0, grid.K - 2)
    frac = pos - i0
    W[np.arange(X0.size), i0] = (1.0 - frac) / grid.h
    W[np.arange(X0.size), i0 + 1] = frac / grid.h
    pad = np.zeros((X0.size, grid.N)); pad[:, :grid.K] = W
    What = rfft(pad, axis=1)

    C = (grid.mult[None, :] * np.real(What * np.conj(Ghat))) * (grid.h / grid.N)
    return np.exp(off) * (C @ H)


# ----------------------------------------------------------------------
# Diagnostic
# ----------------------------------------------------------------------

def grid_report(X_pairs, T, sigma_grid, nu_grid, grid, final_threshold=0.02,
                f2_mean=0.15, lower_true=1e-4, upper_true=1.0 - 1e-9,
                prior_mode="exponential"):
    """Report the two things that can silently break a fixed grid: under-resolved
    features and mass leaking off the edges."""
    nu_max, sd_min, sd_max = nu_grid.max(), sigma_grid.min() * math.sqrt(T), \
        sigma_grid.max() * math.sqrt(T)
    print(f"  {grid}")
    print(f"  points across narrowest beta width : {(2/math.sqrt(nu_max))/grid.h:.1f}")
    print(f"  points across narrowest BM kernel  : {sd_min/grid.h:.1f}")
    print(f"  edge padding in units of sd_max    : "
          f"{min(-12 - grid.x_lo, grid.X[-1] - 12)/sd_max:.1f}")
    q0 = expit(X_pairs[:, 0])
    logW = _log_initial_weight(q0, grid.q, nu_grid.min(), f2_mean,
                               lower_true, upper_true, prior_mode)
    Wn = np.exp(logW - logW.max(axis=1, keepdims=True))
    edge = (Wn[:, :5].sum() + Wn[:, -5:].sum()) / Wn.sum()
    print(f"  fraction of outer weight at edges  : {edge:.2e}")

In [5]:
"""
Both integrals are moved onto ONE fixed uniform grid in logit space, and the
inner (latent-final) integral is recognised as a Gaussian convolution, which is
then done exactly in Fourier space. Nothing about the quadrature depends on
sigma_x, so the log-likelihood surface is smooth by construction.

    p(qT_obs | q0_obs) = num / Z

    num = int w(X0) * S(X0) dX0,     S = ( g_T  *  phi_sd )   (convolution)
    Z   = int w(X0) dX0
    w(X0) = g_0(q0_obs | expit(X0)) * pi(expit(X0)) * expit(X0)(1-expit(X0))

The Jacobian factor q(1-q) appears because pi is a density in q while the grid
and the Brownian transition live in X = logit(q).

Cost per (sigma, nu): one matrix-vector product. The beta evaluations -- the
expensive part -- are hoisted out of the sigma loop entirely.
"""
__all__ = ["LogitGrid", "make_grid", "loglik_grid_spectral", "inner_integral_grid"]


# ----------------------------------------------------------------------
# Grid construction
# ----------------------------------------------------------------------

class LogitGrid:
    """Uniform grid X_k on [x_lo, x_hi] with spacing h, plus its rfft plan size."""

    def __init__(self, x_lo, x_hi, h):
        self.K = int(math.ceil((x_hi - x_lo) / h)) + 1
        self.h = h
        self.x_lo = x_lo
        self.X = x_lo + h * np.arange(self.K)
        self.q = expit(self.X)
        # zero-pad to at least 2K so the circular convolution is a linear one
        self.N = next_fast_len(2 * self.K)
        self.nfreq = self.N // 2 + 1
        self.omega = 2.0 * math.pi * np.arange(self.nfreq) / (self.N * h)
        # Hermitian-symmetry multipliers for a full-spectrum Parseval sum
        self.mult = np.full(self.nfreq, 2.0)
        self.mult[0] = 1.0
        if self.N % 2 == 0:
            self.mult[-1] = 1.0

    def __repr__(self):
        return (f"LogitGrid(range=[{self.x_lo:.1f}, {self.X[-1]:.1f}], "
                f"h={self.h:.2e}, K={self.K}, N_fft={self.N})")


def make_grid(nu_max, sigma_max, T, sigma_min=None, points_per_width=8.0,
              x_core=(-12.0, 12.0), pad_sds=9.0, verbose=False):
    """
    Choose grid spacing and extent from the two length scales in the problem.

    Resolution is set by the narrowest feature: the beta measurement likelihood,
    whose width in logit space is ~1/sqrt(nu*q(1-q)) >= 2/sqrt(nu).
    Extent is set by the widest: the Brownian kernel, sd = sigma_max*sqrt(T),
    padded so the tails of censored (non-decaying) integrands are negligible.
    """
    T = np.asarray(T, dtype=float)
    T_max, T_min = float(T.max()), float(T.min())
    sd_max = sigma_max * math.sqrt(T_max)
    h = (2.0 / math.sqrt(nu_max)) / points_per_width
    if sigma_min is not None:
        sd_min = sigma_min * math.sqrt(T_min)
        if sd_min < 4.0 * h:
            h = sd_min / 4.0  # also resolve the narrowest Brownian kernel
    g = LogitGrid(x_core[0] - pad_sds * sd_max, x_core[1] + pad_sds * sd_max, h)
    if verbose:
        print(f"  {g}")
    return g


# ----------------------------------------------------------------------
# Per-nu precomputation (independent of sigma_x)
# ----------------------------------------------------------------------

def _log_final_emission(qT_obs, q, nu, lower, upper):
    """(P, K) log g(qT_obs | q_k), censored below `lower` and above `upper`."""
    a, b = nu * q, nu * (1.0 - q)
    lnB = betaln(a, b)
    below, above = qT_obs < lower, qT_obs > upper
    inr = ~(below | above)
    out = np.empty((qT_obs.size, q.size))
    if inr.any():
        y = qT_obs[inr][:, None]
        out[inr] = (a - 1.0) * np.log(y) + (b - 1.0) * np.log1p(-y) - lnB
    if below.any():
        out[below] = np.log(np.maximum(betainc(a, b, lower), 1e-300))
    if above.any():
        # 1 - I_x(a,b) = I_{1-x}(b,a): the symmetric form keeps precision
        out[above] = np.log(np.maximum(betainc(b, a, 1.0 - upper), 1e-300))
    return out


def _log_initial_weight(q0_obs, q, nu, f2_mean, lower_true, upper_true, prior_mode):
    """(P, K) log[ g_0(q0_obs | q_k) * pi(q_k) * q_k(1-q_k) ], -inf outside support."""
    a, b = nu * q, nu * (1.0 - q)
    y = q0_obs[:, None]
    log_g0 = (a - 1.0) * np.log(y) + (b - 1.0) * np.log1p(-y) - betaln(a, b)

    with np.errstate(divide="ignore"):
        log_jac = np.log(q) + np.log1p(-q)
        if prior_mode == "exponential":
            log_pi = -q / f2_mean
        elif prior_mode in ("flat", "uniform", "none"):
            log_pi = np.zeros_like(q)
        elif prior_mode == "flat_logit":
            log_pi = -log_jac  # flat in X; the Jacobian cancels exactly
        else:
            raise ValueError("prior_mode must be one of "
                             "{'exponential','flat','flat_logit','plugin'}.")

    row = log_pi + log_jac
    row = np.where((q >= lower_true) & (q < upper_true), row, -np.inf)
    return log_g0 + row[None, :]


def _rows_to_spectrum(logM, grid):
    """Exponentiate rows with a per-row max removed, zero-pad, rfft."""
    off = np.max(logM, axis=1)
    off = np.where(np.isfinite(off), off, 0.0)
    M = np.exp(logM - off[:, None])
    M[~np.isfinite(M)] = 0.0
    pad = np.zeros((M.shape[0], grid.N))
    pad[:, :grid.K] = M
    return rfft(pad, axis=1), off


class _NuCache:
    """Everything that depends on nu but not on sigma_x."""

    def __init__(self, X_pairs, nu, grid, final_threshold, f2_mean,
                 lower_true, upper_true, prior_mode, normalise_posterior):
        q0_obs = expit(X_pairs[:, 0])
        qT_obs = expit(X_pairs[:, 1])
        lower, upper = final_threshold, 1.0 - final_threshold

        logG = _log_final_emission(qT_obs, grid.q, nu, lower, upper)
        Ghat, self.g_off = _rows_to_spectrum(logG, grid)

        if prior_mode == "plugin":
            # collapse the outer integral to a point mass at X0_obs, implemented
            # as a linear-interpolation delta so the same machinery applies
            W = np.zeros((q0_obs.size, grid.K))
            pos = (X_pairs[:, 0] - grid.x_lo) / grid.h
            i0 = np.clip(np.floor(pos).astype(int), 0, grid.K - 2)
            frac = pos - i0
            W[np.arange(W.shape[0]), i0] = (1.0 - frac) / grid.h
            W[np.arange(W.shape[0]), i0 + 1] = frac / grid.h
            pad = np.zeros((W.shape[0], grid.N))
            pad[:, :grid.K] = W
            What = rfft(pad, axis=1)
            self.w_off = np.zeros(q0_obs.size)
            self.log_Z = np.zeros(q0_obs.size)
        else:
            logW = _log_initial_weight(q0_obs, grid.q, nu, f2_mean,
                                       lower_true, upper_true, prior_mode)
            What, self.w_off = _rows_to_spectrum(logW, grid)
            # Zs is computed from the *scaled* rows, so log Z = log Zs + w_off
            Zs = grid.h * np.exp(logW - self.w_off[:, None]).sum(axis=1)
            self.log_Z = (np.log(Zs) + self.w_off if normalise_posterior
                          else np.zeros_like(Zs))

        # Parseval kernel: num = (h/N) * sum_j mult_j * Re[What_j conj(Ghat_j)] * H_j
        self.C = (grid.mult[None, :] *
                  np.real(What * np.conj(Ghat))) * (grid.h / grid.N)
        # constant offset that survives the num/Z ratio
        self.log_off = self.g_off + self.w_off - self.log_Z


# ----------------------------------------------------------------------
# Main entry point -- drop-in for loglik_grid
# ----------------------------------------------------------------------

def loglik_grid_spectral(
    X_pairs, T, sigma_grid, nu_grid,
    final_threshold=0.02, f2_mean=0.15, lower_true=1e-4, upper_true=1.0 - 1e-9,
    prior_mode="exponential", normalise_posterior=True,
    grid=None, points_per_width=8.0, eps=1e-300, return_grid=False,
):
    """
    Total log-likelihood over all pairs on a (sigma_x, nu) grid.

    Signature matches loglik_grid; `n_outer`/`n_hermite` are replaced by the
    single `points_per_width` knob (raise it to tighten the tolerance).

    T may be a scalar (all pairs share an interval) or an array of length P
    giving each pair its own interval. T enters only through sd^2 = sigma^2 * T
    in the Gaussian transfer function, so per-pair T costs nothing extra when
    T takes few distinct values, and one extra pass over sigma otherwise.
    """
    X_pairs = np.asarray(X_pairs, dtype=float)
    sigma_grid = np.asarray(sigma_grid, dtype=float)
    nu_grid = np.asarray(nu_grid, dtype=float)
    P = X_pairs.shape[0]

    T = np.asarray(T, dtype=float)
    if T.ndim == 0:
        T_vec = np.full(P, float(T))
    elif T.shape == (P,):
        T_vec = T
    else:
        raise ValueError(f"T must be a scalar or an array of length P={P}; "
                         f"got shape {T.shape}.")
    if np.any(T_vec <= 0):
        raise ValueError("all T must be strictly positive.")

    if grid is None:
        grid = make_grid(nu_grid.max(), sigma_grid.max(), T_vec.max(),
                         sigma_min=sigma_grid.min(),
                         points_per_width=points_per_width)

    om2 = grid.omega ** 2
    uT, inv = np.unique(T_vec, return_inverse=True)
    grouped = uT.size <= 16          # few distinct intervals -> BLAS-3 per group
    if grouped:
        idx_by_T = [np.flatnonzero(inv == g) for g in range(uT.size)]
        # (nfreq, S) transfer function per distinct interval
        H_by_T = [np.exp(-0.5 * np.outer(om2, sigma_grid ** 2 * Tg)) for Tg in uT]

    logL = np.empty((sigma_grid.size, nu_grid.size))
    for j, nu in enumerate(nu_grid):
        cache = _NuCache(X_pairs, nu, grid, final_threshold, f2_mean,
                         lower_true, upper_true, prior_mode, normalise_posterior)
        ratio = np.empty((P, sigma_grid.size))
        if grouped:
            for idx, H in zip(idx_by_T, H_by_T):
                ratio[idx] = cache.C[idx] @ H
        else:
            for i, s in enumerate(sigma_grid):
                Hp = np.exp(-0.5 * np.outer(s ** 2 * T_vec, om2))   # (P, nfreq)
                ratio[:, i] = np.einsum("pj,pj->p", cache.C, Hp)
        logL[:, j] = np.log(np.maximum(ratio, eps)).sum(axis=0) \
            + cache.log_off.sum()

    return (logL, grid) if return_grid else logL


def inner_integral_grid(X0, qT_obs, sigma_x, T, nu, final_threshold=0.02,
                        grid=None, points_per_width=8.0):
    """Standalone inner integral E[g(qT_obs | X_T)], for validation."""
    X0 = np.atleast_1d(np.asarray(X0, float))
    if grid is None:
        grid = make_grid(nu, sigma_x, T, sigma_min=sigma_x,
                         points_per_width=points_per_width)
    lower, upper = final_threshold, 1.0 - final_threshold
    logG = _log_final_emission(np.atleast_1d(qT_obs), grid.q, nu, lower, upper)
    Ghat, off = _rows_to_spectrum(logG, grid)
    H = np.exp(-0.5 * (grid.omega * sigma_x * math.sqrt(T)) ** 2)

    W = np.zeros((X0.size, grid.K))
    pos = (X0 - grid.x_lo) / grid.h
    i0 = np.clip(np.floor(pos).astype(int), 0, grid.K - 2)
    frac = pos - i0
    W[np.arange(X0.size), i0] = (1.0 - frac) / grid.h
    W[np.arange(X0.size), i0 + 1] = frac / grid.h
    pad = np.zeros((X0.size, grid.N)); pad[:, :grid.K] = W
    What = rfft(pad, axis=1)

    C = (grid.mult[None, :] * np.real(What * np.conj(Ghat))) * (grid.h / grid.N)
    return np.exp(off) * (C @ H)


# ----------------------------------------------------------------------
# Diagnostic
# ----------------------------------------------------------------------

def grid_report(X_pairs, T, sigma_grid, nu_grid, grid, final_threshold=0.02,
                f2_mean=0.15, lower_true=1e-4, upper_true=1.0 - 1e-9,
                prior_mode="exponential"):
    """Report the two things that can silently break a fixed grid: under-resolved
    features and mass leaking off the edges."""
    T = np.asarray(T, dtype=float)
    nu_max = nu_grid.max()
    sd_min = sigma_grid.min() * math.sqrt(float(T.min()))
    sd_max = sigma_grid.max() * math.sqrt(float(T.max()))
    print(f"  {grid}")
    print(f"  points across narrowest beta width : {(2/math.sqrt(nu_max))/grid.h:.1f}")
    print(f"  points across narrowest BM kernel  : {sd_min/grid.h:.1f}")
    print(f"  edge padding in units of sd_max    : "
          f"{min(-12 - grid.x_lo, grid.X[-1] - 12)/sd_max:.1f}")
    q0 = expit(X_pairs[:, 0])
    logW = _log_initial_weight(q0, grid.q, nu_grid.min(), f2_mean,
                               lower_true, upper_true, prior_mode)
    Wn = np.exp(logW - logW.max(axis=1, keepdims=True))
    edge = (Wn[:, :5].sum() + Wn[:, -5:].sum()) / Wn.sum()
    print(f"  fraction of outer weight at edges  : {edge:.2e}")

### Conditional mock 

#### flat prior

In [ ]:
import os

# ---- Grids ----
sigma_grid = np.linspace(0.05, 2.2, 60)
nu_grid    = np.logspace(-1, 4, 60)
base       = "heatmap_results/"
T_days     = 24 / 24.0   # 24 h expressed in days

# ---- nu = 400 noisy datasets ----
NU_400 = 400

dataset_sigma_06_nu400 = add_beta_measurement_noise(dataset_sigma_06_small, NU_400, seed=1200)
dataset_sigma_10_nu400 = add_beta_measurement_noise(dataset_sigma_10_small, NU_400, seed=1200)
dataset_sigma_14_nu400 = add_beta_measurement_noise(dataset_sigma_14_small, NU_400, seed=1200)

# Filter: observed initial must lie in [INIT_FILTER_LOWER, INIT_FILTER_UPPER]
kept_06_nu400, _ = select_calibrated_trajectories(
    dataset_sigma_06_nu400, INIT_FILTER_LOWER, INIT_FILTER_UPPER, N_KEEP, "sigma_06_nu400")
kept_10_nu400, _ = select_calibrated_trajectories(
    dataset_sigma_10_nu400, INIT_FILTER_LOWER, INIT_FILTER_UPPER, N_KEEP, "sigma_10_nu400")
kept_14_nu400, _ = select_calibrated_trajectories(
    dataset_sigma_14_nu400, INIT_FILTER_LOWER, INIT_FILTER_UPPER, N_KEEP, "sigma_14_nu400")

# Logit-space pairs for inference
def _to_X_pairs(kept, t, T):
    return logit_transform(sample_initial_final(kept, t, T))

X_pairs_06_nu400 = _to_X_pairs(kept_06_nu400, t, T_days)
X_pairs_10_nu400 = _to_X_pairs(kept_10_nu400, t, T_days)
X_pairs_14_nu400 = _to_X_pairs(kept_14_nu400, t, T_days)

# nu = 10 logit-space pairs (already computed above via X_samples_all_measnoise)
X_pairs_06_nu10 = X_samples_all_measnoise["sigma_06"][24]
X_pairs_10_nu10 = X_samples_all_measnoise["sigma_10"][24]
X_pairs_14_nu10 = X_samples_all_measnoise["sigma_14"][24]


sigma_06_nu400: 175/200 passed the initial filter (rate 0.88); using the first 50.
sigma_10_nu400: 177/200 passed the initial filter (rate 0.89); using the first 50.
sigma_14_nu400: 170/200 passed the initial filter (rate 0.85); using the first 50.
Computing mock_sigma_06_logL_nu400.npy ...
  saved.
Computing mock_sigma_10_logL_nu400.npy ...
  saved.
Computing mock_sigma_14_logL_nu400.npy ...
  saved.
Computing mock_sigma_06_logL_nu10.npy ...
  saved.
Computing mock_sigma_10_logL_nu10.npy ...
  saved.
Computing mock_sigma_14_logL_nu10.npy ...
  saved.


In [ ]:
grid = make_grid(nu_grid.max(), sigma_grid.max(), T_days, sigma_min=sigma_grid.min())
grid_report(X_pairs_06_nu400, T_days, sigma_grid, nu_grid, grid)   # check before trusting it
logL = loglik_grid_spectral(X_pairs_06_nu400, T_days, sigma_grid, nu_grid,
                            prior_mode="flat", grid=grid)

  LogitGrid(range=[-31.8, 31.8], h=2.50e-03, K=25441, N_fft=51030)
  points across narrowest beta width : 8.0
  points across narrowest BM kernel  : 20.0
  edge padding in units of sd_max    : 9.0
  fraction of outer weight at edges  : 0.00e+00


In [ ]:

# Data generated with init_dist="flat"\
def _compute_or_load_flat(fname, X_pairs):
    if os.path.exists(fname):
        print(f"Loading  {os.path.basename(fname)}")
        return np.load(fname)
    print(f"Computing {os.path.basename(fname)} ...")
    logL = loglik_grid_spectral(
        X_pairs, T_days, sigma_grid, nu_grid,
        prior_mode="flat", grid=grid
    )
    np.save(fname, logL)
    print(f"  saved.")
    return logL


logL_06_nu400 = _compute_or_load_flat(base + "mock_sigma_06_logL_nu400_flat.npy", X_pairs_06_nu400)
logL_10_nu400 = _compute_or_load_flat(base + "mock_sigma_10_logL_nu400_flat.npy", X_pairs_10_nu400)
logL_14_nu400 = _compute_or_load_flat(base + "mock_sigma_14_logL_nu400_flat.npy", X_pairs_14_nu400)

logL_06_nu10  = _compute_or_load_flat(base + "mock_sigma_06_logL_nu10_flat.npy",  X_pairs_06_nu10)
logL_10_nu10  = _compute_or_load_flat(base + "mock_sigma_10_logL_nu10_flat.npy",  X_pairs_10_nu10)
logL_14_nu10  = _compute_or_load_flat(base + "mock_sigma_14_logL_nu10_flat.npy",  X_pairs_14_nu10)


Computing mock_sigma_06_logL_nu400_flat.npy ...
  saved.
Computing mock_sigma_10_logL_nu400_flat.npy ...
  saved.
Computing mock_sigma_14_logL_nu400_flat.npy ...
  saved.
Computing mock_sigma_06_logL_nu10_flat.npy ...
  saved.
Computing mock_sigma_10_logL_nu10_flat.npy ...
  saved.
Computing mock_sigma_14_logL_nu10_flat.npy ...
  saved.


#### Exponential prior 

In [201]:
# Data pairs are the same as those used for the flat prior (from cell 5 above).
X_pairs_06_nu400 = _to_X_pairs(kept_06_nu400, t, T_days)
X_pairs_10_nu400 = _to_X_pairs(kept_10_nu400, t, T_days)
X_pairs_14_nu400 = _to_X_pairs(kept_14_nu400, t, T_days)

X_pairs_06_nu10 = X_samples_all_measnoise["sigma_06"][24]
X_pairs_10_nu10 = X_samples_all_measnoise["sigma_10"][24]
X_pairs_14_nu10 = X_samples_all_measnoise["sigma_14"][24]

# ---- Exponential prior matching the data-generating distribution ----
# Data generated with init_dist="exponential", f2_mean=0.15
# => Exponential(scale=0.15): mean = 0.15, variance = 0.15^2 = 0.0225
F2_MEAN_PRIOR = 0.15


def _compute_or_load_exp(fname, X_pairs):
    if os.path.exists(fname):
        print(f"Loading  {os.path.basename(fname)}")
        return np.load(fname)
    print(f"Computing {os.path.basename(fname)} ...")
    logL = loglik_grid_spectral(
        X_pairs, T_days, sigma_grid, nu_grid,
        final_threshold=FINAL_CENSOR_THRESHOLD,
        prior_mode="exponential",
        f2_mean=F2_MEAN_PRIOR,grid=grid
    )
    np.save(fname, logL)
    print(f"  saved.")
    return logL

logL_06_nu400_exp = _compute_or_load_exp(base + "mock_sigma_06_logL_exp_nu400.npy", X_pairs_06_nu400)
logL_10_nu400_exp = _compute_or_load_exp(base + "mock_sigma_10_logL_exp_nu400.npy", X_pairs_10_nu400)
logL_14_nu400_exp = _compute_or_load_exp(base + "mock_sigma_14_logL_exp_nu400.npy", X_pairs_14_nu400)

logL_06_nu10_exp  = _compute_or_load_exp(base + "mock_sigma_06_logL_exp_nu10.npy",  X_pairs_06_nu10)
logL_10_nu10_exp  = _compute_or_load_exp(base + "mock_sigma_10_logL_exp_nu10.npy",  X_pairs_10_nu10)
logL_14_nu10_exp  = _compute_or_load_exp(base + "mock_sigma_14_logL_exp_nu10.npy",  X_pairs_14_nu10)


Computing mock_sigma_06_logL_exp_nu400.npy ...
  saved.
Computing mock_sigma_10_logL_exp_nu400.npy ...
  saved.
Computing mock_sigma_14_logL_exp_nu400.npy ...
  saved.
Computing mock_sigma_06_logL_exp_nu10.npy ...
  saved.
Computing mock_sigma_10_logL_exp_nu10.npy ...
  saved.
Computing mock_sigma_14_logL_exp_nu10.npy ...
  saved.


In [205]:
grid = make_grid(nu_grid.max(), sigma_grid.max(), T_days, sigma_min=sigma_grid.min())
grid_report(X_pairs_06_nu400, T_days, sigma_grid, nu_grid, grid)   # check before trusting it
logL = loglik_grid_spectral(X_pairs_06_nu400, T_days, sigma_grid, nu_grid,
                            final_threshold=FINAL_CENSOR_THRESHOLD, prior_mode="exponential",
                            f2_mean=F2_MEAN_PRIOR,grid=grid)



  LogitGrid(range=[-31.8, 31.8], h=2.50e-03, K=25441, N_fft=51030)
  points across narrowest beta width : 8.0
  points across narrowest BM kernel  : 20.0
  edge padding in units of sd_max    : 9.0
  fraction of outer weight at edges  : 0.00e+00


#### Fitted exponential

In [273]:

def fit_initial_prior_mean_simple(q0_obs, correct_selection=False,
                                  sel_lower=None, sel_upper=None):
    """
    Simplest prior mean: fit an exponential straight to the OBSERVED initials and
    use its mean as the prior mean on the true initial.
    """
    q0_obs = np.asarray(q0_obs, dtype=float)
    if not correct_selection:
        return float(q0_obs.mean())                 # plain exponential MLE
    lo, hi = sel_lower, sel_upper
    if lo is None or hi is None:
        raise ValueError("correct_selection=True needs sel_lower and sel_upper.")
 

In [274]:
q0_obs = expit(X_pairs[:, 0])
f2 = fit_initial_prior_mean_simple(q0_obs)                       # naive: sample mean
f2 = fit_initial_prior_mean_simple(q0_obs, correct_selection=True,
                                   sel_lower=0.05, sel_upper=0.95)

In [275]:
from scipy.optimize import minimize_scalar

def fit_truncated_exponential(data, lower=0.02, upper=0.98):
    data = np.asarray(data, dtype=float)

    # Check support
    if np.any((data < lower) | (data > upper)):
        raise ValueError(
            f"All data must be between {lower} and {upper}."
        )

    n = len(data)

    def neg_loglik(lam):
        if lam <= 0:
            return np.inf

        # Numerically stable truncated exponential log-likelihood
        #
        # pdf =
        # lam * exp[-lam * (x-lower)]
        # ----------------------------------
        # 1 - exp[-lam * (upper-lower)]

        width = upper - lower

        log_norm = np.log(
            -np.expm1(-lam * width)
        )

        loglik = (
            n * np.log(lam)
            - lam * np.sum(data - lower)
            - n * log_norm
        )

        return -loglik

    result = minimize_scalar(
        neg_loglik,
        bounds=(1e-8, 1000),
        method="bounded"
    )

    return result.x


In [277]:
# Data pairs are the same as those used for the flat prior (from cell 5 above).
X_pairs_06_nu400 = _to_X_pairs(kept_06_nu400, t, T_days)
X_pairs_10_nu400 = _to_X_pairs(kept_10_nu400, t, T_days)
X_pairs_14_nu400 = _to_X_pairs(kept_14_nu400, t, T_days)

X_pairs_06_nu10 = X_samples_all_measnoise["sigma_06"][24]
X_pairs_10_nu10 = X_samples_all_measnoise["sigma_10"][24]
X_pairs_14_nu10 = X_samples_all_measnoise["sigma_14"][24]

# ---- Exponential prior fitted from q0 observed  ----

sigma_grid = np.linspace(0.05, 2.2, 60)
nu_grid    = np.logspace(-1, 4, 60)

def _compute_or_load_exp(fname, X_pairs):
    fname = fname.replace(".npy", ".npz")
    if os.path.exists(fname):
        print(f"Loading  {os.path.basename(fname)}")
        d = np.load(fname)
        return d["logL"], float(d["f2_mean"])

    print(f"Computing {os.path.basename(fname)} ...")
    q0_obs   = expit(X_pairs[:, 0])
    lambdaa  = fit_truncated_exponential(q0_obs, lower=0.02, upper=0.98)
    f2_mean  = 1.0 / lambdaa

    logL = loglik_grid_spectral(
        X_pairs, T_days, sigma_grid, nu_grid,
        prior_mode="exponential",
        f2_mean=f2_mean,
    )
    np.savez(
        fname,
        logL=logL,
        f2_mean=f2_mean,
        lambdaa=lambdaa,
        sigma_grid=sigma_grid,
        nu_grid=nu_grid,
    )
    print("  saved.")
    return logL, lambdaa

logL_06_nu400_fitexp, F2_06_nu400 = _compute_or_load_exp(base + "mock_sigma_06_logL_fitexp_nu400.npy", X_pairs_06_nu400)
logL_10_nu400_fitexp, F2_10_nu400 = _compute_or_load_exp(base + "mock_sigma_10_logL_fitexp_nu400.npy", X_pairs_10_nu400)
logL_14_nu400_fitexp, F2_14_nu400 = _compute_or_load_exp(base + "mock_sigma_14_logL_fitexp_nu400.npy", X_pairs_14_nu400)

logL_06_nu10_fitexp, F2_06_nu10 = _compute_or_load_exp(base + "mock_sigma_06_logL_fitexp_nu10.npy",  X_pairs_06_nu10)
logL_10_nu10_fitexp, F2_10_nu10 = _compute_or_load_exp(base + "mock_sigma_10_logL_fitexp_nu10.npy",  X_pairs_10_nu10)
logL_14_nu10_fitexp, F2_14_nu10 = _compute_or_load_exp(base + "mock_sigma_14_logL_fitexp_nu10.npy",  X_pairs_14_nu10)


Computing mock_sigma_06_logL_fitexp_nu400.npz ...
  saved.
Computing mock_sigma_10_logL_fitexp_nu400.npz ...
  saved.
Computing mock_sigma_14_logL_fitexp_nu400.npz ...
  saved.
Computing mock_sigma_06_logL_fitexp_nu10.npz ...
  saved.
Computing mock_sigma_10_logL_fitexp_nu10.npz ...
  saved.
Computing mock_sigma_14_logL_fitexp_nu10.npz ...
  saved.


#### q0_obs = q0_true

In [ ]:
grid = make_grid(nu_grid.max(), sigma_grid.max(), T_days, sigma_min=sigma_grid.min())
grid_report(X_pairs_06_nu400, T_days, sigma_grid, nu_grid, grid)   # check before trusting it
logL = loglik_grid_spectral(X_pairs_06_nu400, T_days, sigma_grid, nu_grid,
                            prior_mode="plugin", grid=grid)


# ---- Exponential prior matching the data-generating distribution ----
# Data generated with init_dist="plugin", f2_mean=0.15
def _compute_or_load_exp(fname, X_pairs):
    if os.path.exists(fname):
        print(f"Loading  {os.path.basename(fname)}")
        return np.load(fname)
    print(f"Computing {os.path.basename(fname)} ...")
    logL = loglik_grid_spectral(
        X_pairs, T_days, sigma_grid, nu_grid,
        prior_mode="plugin", grid=grid
    )
    np.save(fname, logL)
    print(f"  saved.")
    return logL

logL_06_nu400_nom1 = _compute_or_load_exp(base + "mock_sigma_06_logL_nom1_nu400.npy", X_pairs_06_nu400)
logL_10_nu400_nom1 = _compute_or_load_exp(base + "mock_sigma_10_logL_nom1_nu400.npy", X_pairs_10_nu400)
logL_14_nu400_nom1 = _compute_or_load_exp(base + "mock_sigma_14_logL_nom1_nu400.npy", X_pairs_14_nu400)

logL_06_nu10_nom1  = _compute_or_load_exp(base + "mock_sigma_06_logL_nom1_nu10.npy",  X_pairs_06_nu10)
logL_10_nu10_nom1  = _compute_or_load_exp(base + "mock_sigma_10_logL_nom1_nu10.npy",  X_pairs_10_nu10)
logL_14_nu10_nom1  = _compute_or_load_exp(base + "mock_sigma_14_logL_nom1_nu10.npy",  X_pairs_14_nu10)


### Conditional real 

##### VanInsberghe

In [280]:
# --------------------------------------------------
# Load and preprocess VanInsberghe (swine IAV) data
# --------------------------------------------------

# VanInsberghe LOD is 0.005 (as used in supplement.ipynb S4)
LOD = 0.02

vani = pd.read_csv("data/raw_data/table_S3.csv")
vani["DPS1"] = pd.to_numeric(vani["DPS1"], errors="coerce")
vani["DPS2"] = pd.to_numeric(vani["DPS2"], errors="coerce")
vani["DPS1 Frequency"] = pd.to_numeric(vani["DPS1 Frequency"], errors="coerce")
vani["DPS2 Frequency"] = pd.to_numeric(vani["DPS2 Frequency"], errors="coerce")
vani = vani.dropna(subset=["DPS1", "DPS2", "DPS1 Frequency", "DPS2 Frequency"])

# Use Subset == 1 (one iSNV per individual x timepoint pair), matching S4
vani = vani[vani["Subset"] == 1].copy()

vani["T"] = vani["DPS2"] - vani["DPS1"]
vani = vani[vani["T"] > 0].copy()

# Filter for DPS1 Frequency above LOD
vani = vani[vani["DPS1 Frequency"] >= LOD].copy()

# DPS2 Frequency = 0 is below LOD; logit_transform clips to eps
# -> measurement model returns P(obs < LOD | latent q) via the beta CDF
X0_arr = logit_transform(vani["DPS1 Frequency"].values)
XT_arr = logit_transform(vani["DPS2 Frequency"].values)
T_arr  = vani["T"].values

print(f"VanInsberghe dataset: {len(vani)} paired observations (Subset == 1)")
print(f"T range: {T_arr.min():.0f} - {T_arr.max():.0f} days")


VanInsberghe dataset: 86 paired observations (Subset == 1)
T range: 1 - 4 days


In [281]:
sigma_grid = np.linspace(0.2, 5, 60)
nu_grid    = np.logspace(0, 4, 60)

X_pairs = np.column_stack([X0_arr, XT_arr])

grid = make_grid(nu_grid.max(), sigma_grid.max(), T_arr, sigma_min=sigma_grid.min())
grid_report(X_pairs, T_arr, sigma_grid, nu_grid, grid)   # check before trusting it

  LogitGrid(range=[-102.0, 102.0], h=2.50e-03, K=81601, N_fft=163296)
  points across narrowest beta width : 8.0
  points across narrowest BM kernel  : 80.0
  edge padding in units of sd_max    : 9.0
  fraction of outer weight at edges  : 0.00e+00


In [282]:

# --------------------------------------------------
# 2D log-likelihood grid over (sigma_x, nu)
# --------------------------------------------------

print("\nComputing log-likelihood grid ...")
logL_grid = loglik_grid_spectral(
    X_pairs, T_arr, sigma_grid, nu_grid,
    prior_mode="flat", grid=grid
)

max_idx   = np.unravel_index(np.argmax(logL_grid), logL_grid.shape)
sigma_hat = sigma_grid[max_idx[0]]
nu_hat    = nu_grid[max_idx[1]]

print(f"\nMLE sigma_x = {sigma_hat:.4f}")
print(f"MLE nu      = {nu_hat:.4f}  (log10 nu = {np.log10(nu_hat):.4f})")
print(f"Max log-likelihood = {logL_grid[max_idx]:.4f}")

np.save("heatmap_results/vaninsberghe_logL_flat.npy", logL_grid)
print("Saved heatmap_results/vaninsberghe_logL_flat.npy")



Computing log-likelihood grid ...

MLE sigma_x = 2.8034
MLE nu      = 49.5354  (log10 nu = 1.6949)
Max log-likelihood = -56.2262
Saved heatmap_results/vaninsberghe_logL_flat.npy


In [284]:
lambda_hat = fit_truncated_exponential(
    vani["DPS1 Frequency"].values,
    lower=0.02,
    upper=0.98
)

print("\nComputing log-likelihood grid ...")
logL_grid = loglik_grid_spectral(
    X_pairs, T_arr, sigma_grid, nu_grid,
    prior_mode="exponential", grid=grid, f2_mean=1/lambda_hat
)

max_idx   = np.unravel_index(np.argmax(logL_grid), logL_grid.shape)
sigma_hat = sigma_grid[max_idx[0]]
nu_hat    = nu_grid[max_idx[1]]

print(f"\nMLE sigma_x = {sigma_hat:.4f}")
print(f"MLE nu      = {nu_hat:.4f}  (log10 nu = {np.log10(nu_hat):.4f})")
print(f"Max log-likelihood = {logL_grid[max_idx]:.4f}")

np.save("heatmap_results/vaninsberghe_logL_exponential.npy", logL_grid)
print("Saved heatmap_results/vaninsberghe_logL_exponential.npy")


Computing log-likelihood grid ...

MLE sigma_x = 0.2000
MLE nu      = 1.3664  (log10 nu = 0.1356)
Max log-likelihood = -49.3938
Saved heatmap_results/vaninsberghe_logL_exponential.npy


In [285]:
logL_grid = loglik_grid_spectral(
    X_pairs, T_arr, sigma_grid, nu_grid,
    prior_mode="plugin", grid=grid,
)

np.save("heatmap_results/vaninsberghe_logL_nom1.npy", logL_grid)
print("Saved heatmap_results/vaninsberghe_logL_nom1.npy")

Saved heatmap_results/vaninsberghe_logL_nom1.npy


In [286]:
sigma_grid = np.linspace(0.2, 5, 60)
nu_grid    = np.logspace(0, 4, 60)

X_pairs = np.column_stack([X0_arr, XT_arr])

q0_obs   = expit(X0_arr)
lambdaa  = fit_truncated_exponential(q0_obs, lower=0.02, upper=0.98)
f2_mean  = 1.0 / lambdaa

logL = loglik_grid_spectral(
        X_pairs, T_arr, sigma_grid, nu_grid,
        prior_mode="exponential",
        f2_mean=f2_mean,
    )

np.savez(
        "heatmap_results/vaninsberghe_logL_fitexp.npz",
        logL=logL,
        f2_mean=f2_mean,
        lambdaa=lambdaa,
        sigma_grid=sigma_grid,
        nu_grid=nu_grid,
    )
print("  saved.")



  saved.


##### McCrone

In [297]:

# --------------------------------------------------
# Load and preprocess McCrone (IAV) data
# --------------------------------------------------

LOD = 0.02

mcc = pd.read_csv("data/raw_data/IAV-Table_S1.csv")
mcc["DPS1"] = pd.to_numeric(mcc["DPS1"], errors="coerce")
mcc["DPS2"] = pd.to_numeric(mcc["DPS2"], errors="coerce")
mcc["DPS1 Frequency"] = pd.to_numeric(mcc["DPS1 Frequency"], errors="coerce")
mcc["DPS2 Frequency"] = pd.to_numeric(mcc["DPS2 Frequency"], errors="coerce")
mcc = mcc.dropna(subset=["DPS1", "DPS2", "DPS1 Frequency", "DPS2 Frequency"])
mcc["T"] = mcc["DPS2"] - mcc["DPS1"]
mcc = mcc[mcc["T"] > 0].copy()

# IAV-Table_S1.csv already filters DPS1 Frequency >= LOD
# DPS2 Frequency = 0 is encoded as below-LOD; logit_transform clips to eps
# -> measurement model returns P(obs < LOD | latent q) via the beta CDF
X0_arr = logit_transform(mcc["DPS1 Frequency"].values)
XT_arr = logit_transform(mcc["DPS2 Frequency"].values)
T_arr  = mcc["T"].values

print(f"McCrone dataset: {len(mcc)} paired observations")
print(f"T range: {T_arr.min():.0f} - {T_arr.max():.0f} days")


McCrone dataset: 60 paired observations
T range: 1 - 6 days


In [298]:

# --------------------------------------------------
# 2D log-likelihood grid over (sigma_x, nu)
# --------------------------------------------------

sigma_grid = np.linspace(0.2, 3.0, 60)
nu_grid    = np.logspace(0, 4, 60)

X_pairs = np.column_stack([X0_arr, XT_arr])

grid = make_grid(nu_grid.max(), sigma_grid.max(), T_arr, sigma_min=sigma_grid.min())
grid_report(X_pairs, T_arr, sigma_grid, nu_grid, grid)   # check before trusting it

  LogitGrid(range=[-78.1, 78.1], h=2.50e-03, K=62510, N_fft=125440)
  points across narrowest beta width : 8.0
  points across narrowest BM kernel  : 80.0
  edge padding in units of sd_max    : 9.0
  fraction of outer weight at edges  : 0.00e+00


In [220]:
# --------------------------------------------------
# 2D log-likelihood grid over (sigma_x, nu)
# --------------------------------------------------

print("\nComputing log-likelihood grid ...")
logL_grid = loglik_grid_spectral(
    X_pairs, T_arr, sigma_grid, nu_grid,
    prior_mode="flat", grid=grid
)

max_idx   = np.unravel_index(np.argmax(logL_grid), logL_grid.shape)
sigma_hat = sigma_grid[max_idx[0]]
nu_hat    = nu_grid[max_idx[1]]

print(f"\nMLE sigma_x = {sigma_hat:.4f}")
print(f"MLE nu      = {nu_hat:.4f}  (log10 nu = {np.log10(nu_hat):.4f})")
print(f"Max log-likelihood = {logL_grid[max_idx]:.4f}")

np.save("heatmap_results/mccrone_logL_flat.npy", logL_grid)
print("Saved heatmap_results/mccrone_logL_flat.npy")



Computing log-likelihood grid ...

MLE sigma_x = 1.5288
MLE nu      = 10000.0000  (log10 nu = 4.0000)
Max log-likelihood = -4.6580
Saved heatmap_results/mccrone_logL_flat.npy


In [240]:
lambda_hat = fit_truncated_exponential(
    mcc["DPS1 Frequency"].values,
    lower=0.02,
    upper=0.98
)

print("\nComputing log-likelihood grid ...")
logL_grid = loglik_grid_spectral(
    X_pairs, T_arr, sigma_grid, nu_grid,
    prior_mode="exponential", grid=grid, f2_mean=1/lambda_hat
)

max_idx   = np.unravel_index(np.argmax(logL_grid), logL_grid.shape)
sigma_hat = sigma_grid[max_idx[0]]
nu_hat    = nu_grid[max_idx[1]]

print(f"\nMLE sigma_x = {sigma_hat:.4f}")
print(f"MLE nu      = {nu_hat:.4f}  (log10 nu = {np.log10(nu_hat):.4f})")
print(f"Max log-likelihood = {logL_grid[max_idx]:.4f}")

np.save("heatmap_results/mccrone_logL_exponential.npy", logL_grid)
print("Saved heatmap_results/mccrone_logL_exponential.npy")

logL_grid = loglik_grid_spectral(
    X_pairs, T_arr, sigma_grid, nu_grid,
    prior_mode="plugin", grid=grid,
)

np.save("heatmap_results/mccrone_logL_nom1.npy", logL_grid)
print("Saved heatmap_results/mccrone_logL_nom1.npy")



Computing log-likelihood grid ...

MLE sigma_x = 1.5288
MLE nu      = 10000.0000  (log10 nu = 4.0000)
Max log-likelihood = -4.6426
Saved heatmap_results/mccrone_logL_exponential.npy
Saved heatmap_results/mccrone_logL_nom1.npy


In [289]:
sigma_grid = np.linspace(0.2, 3.0, 60)
nu_grid    = np.logspace(0, 4, 60)

X_pairs = np.column_stack([X0_arr, XT_arr])

q0_obs   = expit(X0_arr)
lambdaa  = fit_truncated_exponential(q0_obs, lower=0.02, upper=0.98)
f2_mean  = 1.0 / lambdaa

logL = loglik_grid_spectral(
        X_pairs, T_arr, sigma_grid, nu_grid,
        prior_mode="exponential",
        f2_mean=f2_mean,
    )

np.savez(
        "heatmap_results/mccrone_logL_fitexp.npz",
        logL=logL,
        f2_mean=f2_mean,
        lambdaa=lambdaa,
        sigma_grid=sigma_grid,
        nu_grid=nu_grid,
    )
print("  saved.")

  saved.


In [299]:
sigma_grid = np.linspace(0.2, 5.0, 60)
nu_grid    = np.logspace(0, 4, 60)

X_pairs = np.column_stack([X0_arr, XT_arr])

q0_obs   = expit(X0_arr)
lambdaa  = fit_truncated_exponential(q0_obs, lower=0.02, upper=0.98)
f2_mean  = 1.0 / lambdaa

logL = loglik_grid_spectral(
        X_pairs, T_arr, sigma_grid, nu_grid,
        prior_mode="exponential",
        f2_mean=f2_mean,
    )

np.savez(
        "heatmap_results/mccrone_logL_fitexp_05.npz",
        logL=logL,
        f2_mean=f2_mean,
        lambdaa=lambdaa,
        sigma_grid=sigma_grid,
        nu_grid=nu_grid,
    )
print("  saved.")


logL_grid = loglik_grid_spectral(
    X_pairs, T_arr, sigma_grid, nu_grid,
    prior_mode="plugin", grid=grid,
)

np.save("heatmap_results/mccrone_logL_nom1_05.npy", logL_grid)
print("Saved heatmap_results/mccrone_logL_nom1_05.npy")

print("\nComputing log-likelihood grid ...")
logL_grid = loglik_grid_spectral(
    X_pairs, T_arr, sigma_grid, nu_grid,
    prior_mode="flat", grid=grid
)

np.save("heatmap_results/mccrone_logL_flat_05.npy", logL_grid)
print("Saved heatmap_results/mccrone_logL_flat_05.npy")

  saved.
Saved heatmap_results/mccrone_logL_nom1_05.npy

Computing log-likelihood grid ...
Saved heatmap_results/mccrone_logL_flat_05.npy


##### Tonkinhill

In [300]:
LOD = 0.02

ton = pd.read_csv("data/Table_S2_final.csv")
ton["First iSNV frequency"] = pd.to_numeric(ton["First iSNV frequency"], errors="coerce")
ton["Second iSNV frequency"] = pd.to_numeric(ton["Second iSNV frequency"], errors="coerce")
ton["Time between samples (days)"] = pd.to_numeric(
    ton["Time between samples (days)"], errors="coerce"
)
ton = ton.dropna(
    subset=["First iSNV frequency", "Second iSNV frequency", "Time between samples (days)"]
)
ton = ton[ton["Time between samples (days)"] > 0].copy()

# Table_S2_final.csv already filtered for First iSNV frequency > LOD
# Second iSNV frequency may be below LOD; logit_transform clips to eps
# -> measurement model returns P(obs < LOD | latent q) via the beta CDF
X0_arr = logit_transform(ton["First iSNV frequency"].values)
XT_arr = logit_transform(ton["Second iSNV frequency"].values)
T_arr  = ton["Time between samples (days)"].values

print(f"TonkinHill dataset: {len(ton)} paired observations")
print(f"T range: {T_arr.min():.0f} - {T_arr.max():.0f} days")


TonkinHill dataset: 54 paired observations
T range: 1 - 3 days


In [301]:
import numpy as np
from scipy.special import expit, logit, log_ndtr

s_test, nu_test = 1, 1e5
x_thr = float(logit(0.02))
X_pairs = np.column_stack([X0_arr, XT_arr])


T_vec   = T_arr  = ton["Time between samples (days)"].values

P       = len(T_vec)

# one shared grid so every row is integrated identically
g = make_grid(nu_test, s_test, T_vec.max(), sigma_min=s_test)

ll_spec = np.array([
    loglik_grid_spectral(
        X_pairs[i:i+1], T_vec[i],
        np.array([s_test]), np.array([nu_test]),
        prior_mode="plugin", final_threshold=0.02, grid=g,
    )[0, 0]
    for i in range(P)
])

# reference, per pair, masked (no nan from the -inf rows)
sd   = s_test * np.sqrt(T_vec)
cens = X_pairs[:, 1] <= x_thr
ll_ref = np.empty(P)
ll_ref[cens] = log_ndtr((x_thr - X_pairs[cens, 0]) / sd[cens])
z    = (X_pairs[~cens, 1] - X_pairs[~cens, 0]) / sd[~cens]
fT   = np.clip(expit(X_pairs[~cens, 1]), 1e-12, 1 - 1e-12)
ll_ref[~cens] = (-0.5 * z**2 - np.log(sd[~cens]) - 0.5 * np.log(2 * np.pi)
                 - np.log(fT) - np.log1p(-fT))

d = ll_spec - ll_ref
print(f"total gap {d.sum():+.3f} nats over {P} pairs\n")
for i in np.argsort(-np.abs(d))[:15]:
    print(f"f0={expit(X_pairs[i,0]):.4f}  fT={expit(X_pairs[i,1]):.4f}  "
          f"T={T_vec[i]:.1f}  ref={ll_ref[i]:+8.3f}  spec={ll_spec[i]:+8.3f}  "
          f"d={d[i]:+7.3f}")

total gap -8.268 nats over 54 pairs

f0=0.5989  fT=0.9825  T=2.0  ref=  -0.490  spec=  -4.992  d= -4.502
f0=0.9785  fT=0.9975  T=1.0  ref=  +2.710  spec=  -0.754  d= -3.464
f0=0.8925  fT=0.9997  T=2.0  ref=  -1.956  spec=  -2.257  d= -0.301
f0=0.4098  fT=0.0055  T=2.0  ref=  -5.065  spec=  -5.064  d= +0.001
f0=0.0315  fT=0.0208  T=1.0  ref=  +2.884  spec=  +2.883  d= -0.000
f0=0.2636  fT=0.0105  T=3.0  ref=  -3.014  spec=  -3.014  d= +0.000
f0=0.0321  fT=0.0215  T=1.0  ref=  +2.857  spec=  +2.857  d= -0.000
f0=0.0263  fT=0.0223  T=1.0  ref=  +2.891  spec=  +2.890  d= -0.000
f0=0.0205  fT=0.0225  T=1.0  ref=  +2.891  spec=  +2.891  d= -0.000
f0=0.0634  fT=0.0412  T=1.0  ref=  +2.210  spec=  +2.209  d= -0.000
f0=0.0602  fT=0.0427  T=1.0  ref=  +2.213  spec=  +2.213  d= -0.000
f0=0.0271  fT=0.0340  T=1.0  ref=  +2.470  spec=  +2.470  d= -0.000
f0=0.6123  fT=0.1391  T=1.0  ref=  -1.396  spec=  -1.396  d= +0.000
f0=0.0247  fT=0.0244  T=2.0  ref=  +2.474  spec=  +2.474  d= -0.000
f0=0.0495  

In [ ]:
sigma_grid = np.linspace(0.2, 3.0, 60)
nu_grid    = np.logspace(0, 4, 60)

X_pairs = np.column_stack([X0_arr, XT_arr])

grid = make_grid(nu_grid.max(), sigma_grid.max(), T_arr, sigma_min=sigma_grid.min())
grid_report(X_pairs, T_arr, sigma_grid, nu_grid, grid)   # check before trusting it

  LogitGrid(range=[-89.9, 89.9], h=2.50e-03, K=71955, N_fft=144000)
  points across narrowest beta width : 8.0
  points across narrowest BM kernel  : 80.0
  edge padding in units of sd_max    : 9.0
  fraction of outer weight at edges  : 0.00e+00


In [267]:
print("\nComputing log-likelihood grid ...")
logL_grid = loglik_grid_spectral(
    X_pairs, T_arr, sigma_grid, nu_grid,
    prior_mode="flat", grid=grid
)

np.save("heatmap_results/tonkinhill_logL_flat.npy", logL_grid)
print("Saved heatmap_results/tonkinhill_logL_flat.npy")

lambda_hat = fit_truncated_exponential(
    ton["First iSNV frequency"].values,
    lower=0.02,
    upper=0.98
)

print("\nComputing log-likelihood grid ...")
logL_grid = loglik_grid_spectral(
    X_pairs, T_arr, sigma_grid, nu_grid,
    prior_mode="exponential", grid=grid, f2_mean=1/lambda_hat
)

max_idx   = np.unravel_index(np.argmax(logL_grid), logL_grid.shape)
sigma_hat = sigma_grid[max_idx[0]]
nu_hat    = nu_grid[max_idx[1]]

print(f"\nMLE sigma_x = {sigma_hat:.4f}")
print(f"MLE nu      = {nu_hat:.4f}  (log10 nu = {np.log10(nu_hat):.4f})")
print(f"Max log-likelihood = {logL_grid[max_idx]:.4f}")

np.save("heatmap_results/tonkinhill_logL_exponential.npy", logL_grid)
print("Saved heatmap_results/tonkinhill_logL_exponential.npy")

logL_grid = loglik_grid_spectral(
    X_pairs, T_arr, sigma_grid, nu_grid,
    prior_mode="plugin", grid=grid,
)

np.save("heatmap_results/tonkinhill_logL_nom1.npy", logL_grid)
print("Saved heatmap_results/tonkinhill_logL_nom1.npy")



Computing log-likelihood grid ...
Saved heatmap_results/tonkinhill_logL_flat.npy

Computing log-likelihood grid ...

MLE sigma_x = 1.1492
MLE nu      = 10000.0000  (log10 nu = 4.0000)
Max log-likelihood = 23.8655
Saved heatmap_results/tonkinhill_logL_exponential.npy
Saved heatmap_results/tonkinhill_logL_nom1.npy


In [294]:
sigma_grid = np.linspace(0.2, 3.0, 60)
nu_grid    = np.logspace(0, 4, 60)

X_pairs = np.column_stack([X0_arr, XT_arr])

q0_obs   = expit(X0_arr)
lambdaa  = fit_truncated_exponential(q0_obs, lower=0.02, upper=0.98)
f2_mean  = 1.0 / lambdaa

logL = loglik_grid_spectral(
        X_pairs, T_arr, sigma_grid, nu_grid,
        prior_mode="exponential",
        f2_mean=f2_mean,
    )

np.savez(
        "heatmap_results/tonkinhill_logL_fitexp.npz",
        logL=logL,
        f2_mean=f2_mean,
        lambdaa=lambdaa,
        sigma_grid=sigma_grid,
        nu_grid=nu_grid,
    )
print("  saved.")

  saved.
